# CC3074 – Modelización y Simulación
## A* (A-estrella)

**Universidad del Valle de Guatemala**

Este notebook estudia A* desde cero. Primero se define cada elemento, después se resuelve el ejemplo de la presentación a mano y finalmente se implementa en Python.

### Objetivos
- Comprender qué es un grafo.
- Diferenciar costo real y heurística.
- Comprender la función de evaluación de A*.
- Resolver un ejemplo manualmente.
- Implementar A* sin ocultar su lógica.
- Interpretar el camino y su costo.

## 1. Antes de A*: ¿qué es un grafo?

Un **grafo** está formado por:

- **Nodo o vértice:** representa un lugar, estado o elemento.
- **Arista:** conexión entre dos nodos.
- **Peso o costo:** cantidad asociada a recorrer una arista.

Ejemplo:

```text
A ----4---- B
 \          |
  2         3
   \        |
    C --1-- D
```

El valor `4` puede representar distancia, tiempo, dinero, energía u otra medida de costo.

## 2. ¿Qué problema resuelve A*?

Queremos encontrar un camino desde un nodo de **inicio** hasta un nodo **meta**.

A* combina dos cantidades.

### Costo conocido

$g(n)$ = costo real acumulado desde el inicio hasta el nodo $n$.

### Heurística

$h(n)$ = estimación del costo que todavía falta desde $n$ hasta la meta.

### Evaluación

$$
f(n)=g(n)+h(n)
$$

Donde:

- $n$ = nodo evaluado.
- $g(n)$ = costo conocido.
- $h(n)$ = costo futuro estimado.
- $f(n)$ = valor que A* utiliza para priorizar la exploración.

Una heurística no es el costo real restante: es una **estimación**.

## 3. Ejemplo de la presentación

Conexiones:

```text
S → A = 2
S → B = 4
A → C = 2
A → G = 7
B → G = 5
C → G = 2
```

Heurísticas:

```text
h(S) = 5
h(A) = 4
h(B) = 4
h(C) = 2
h(G) = 0
```

`S` es el inicio y `G` es la meta.

## 4. Resolución a mano: primer paso

Inicialmente:

$$
g(S)=0
$$

$$
f(S)=g(S)+h(S)=0+5=5
$$

Desde `S` podemos ir a `A` o `B`.

### Nodo A

$$
g(A)=0+2=2
$$

$$
f(A)=2+4=6
$$

### Nodo B

$$
g(B)=0+4=4
$$

$$
f(B)=4+4=8
$$

A* explora primero `A`, porque `6 < 8`.

## 5. Resolución a mano: segundo paso

Desde `A` podemos ir a `C` o a `G`.

### A → C

$$
g(C)=g(A)+2=2+2=4
$$

$$
f(C)=4+2=6
$$

### A → G

$$
g(G)=g(A)+7=2+7=9
$$

Como $h(G)=0$:

$$
f(G)=9
$$

Opciones pendientes:

```text
C → f = 6
B → f = 8
G → f = 9
```

A* selecciona `C`.

## 6. Resolución a mano: llegada a la meta

Desde `C` podemos llegar a `G` con costo `2`.

$$
g(G)=g(C)+2=4+2=6
$$

Por tanto, el camino encontrado es:

```text
S → A → C → G
```

Costo total:

$$
2+2+2=6
$$

## 7. Representación del mismo problema en Python

In [ ]:
grafo = {
    "S": [("A", 2), ("B", 4)],
    "A": [("C", 2), ("G", 7)],
    "B": [("G", 5)],
    "C": [("G", 2)],
    "G": []
}

heuristica = {
    "S": 5,
    "A": 4,
    "B": 4,
    "C": 2,
    "G": 0
}

print(grafo)
print(heuristica)

## 8. Cola de prioridad

A* necesita escoger repetidamente el candidato con menor $f(n)$.

Python incluye `heapq`, que implementa una **cola de prioridad**.

Guardaremos elementos con esta idea:

```text
(f, g, nodo, camino)
```

El primer valor es `f`, por lo que el elemento con menor `f` queda con prioridad.

In [ ]:
import heapq

cola = []
heapq.heappush(cola, (6, 2, "A"))
heapq.heappush(cola, (8, 4, "B"))

print("Sale primero:", heapq.heappop(cola))

## 9. Implementación didáctica de A*

In [ ]:
import heapq

def a_estrella(grafo, heuristica, inicio, meta):
    # (f, g, nodo, camino)
    abiertos = [(heuristica[inicio], 0, inicio, [inicio])]

    # Mejor costo g conocido para cada nodo.
    mejor_g = {inicio: 0}

    while abiertos:
        f_actual, g_actual, actual, camino = heapq.heappop(abiertos)

        print(
            f"Explorando {actual} | "
            f"g={g_actual} | h={heuristica[actual]} | f={f_actual}"
        )

        if actual == meta:
            return camino, g_actual

        # Si este registro quedó obsoleto, no lo expandimos.
        if g_actual > mejor_g.get(actual, float("inf")):
            continue

        for vecino, costo in grafo[actual]:
            nuevo_g = g_actual + costo

            if nuevo_g < mejor_g.get(vecino, float("inf")):
                mejor_g[vecino] = nuevo_g
                nuevo_f = nuevo_g + heuristica[vecino]

                heapq.heappush(
                    abiertos,
                    (nuevo_f, nuevo_g, vecino, camino + [vecino])
                )

    return None, None

## 10. Ejecutar A*

In [ ]:
camino, costo = a_estrella(
    grafo,
    heuristica,
    "S",
    "G"
)

print("\nCamino:", camino)
print("Costo:", costo)

### Resultado esperado

```text
Camino: ['S', 'A', 'C', 'G']
Costo: 6
```

La computadora reproduce el mismo razonamiento que hicimos manualmente.

## 11. ¿Qué hace cada variable?

- `abiertos`: candidatos pendientes de explorar.
- `f_actual`: valor $g+h$ del candidato extraído.
- `g_actual`: costo real acumulado hasta ese nodo.
- `actual`: nodo que estamos procesando.
- `camino`: secuencia seguida hasta el nodo actual.
- `mejor_g`: mejor costo real conocido para cada nodo.
- `vecino`: nodo conectado con el actual.
- `costo`: costo de la arista.
- `nuevo_g`: costo acumulado si llegamos al vecino por el camino actual.
- `nuevo_f`: `nuevo_g + heuristica[vecino]`.

## 12. ¿Qué ocurre si la heurística es cero?

Si para todos los nodos usamos:

$$
h(n)=0
$$

entonces:

$$
f(n)=g(n)
$$

A* deja de utilizar información estimada sobre la meta y prioriza únicamente el costo acumulado.

Esta observación ayuda a entender por qué la heurística es la parte que hace que A* sea una búsqueda **informada**.

In [ ]:
heuristica_cero = {nodo: 0 for nodo in heuristica}

camino, costo = a_estrella(
    grafo,
    heuristica_cero,
    "S",
    "G"
)

print("Camino:", camino)
print("Costo:", costo)

## 13. Ejercicio

Modifique el grafo agregando un nodo `D`:

```text
B → D = 1
D → G = 2
```

Asigne:

```text
h(D) = 1
```

Preguntas:

1. ¿Qué camino encuentra A*?
2. ¿Cuál es su costo?
3. Resuélvalo primero a mano.
4. Después confirme el resultado modificando el código.

## 14. Idea final

A* combina:

```text
lo que ya gasté + lo que estimo que falta
```

Matemáticamente:

$$
f(n)=g(n)+h(n)
$$

La calidad de la heurística influye en qué tan útil resulta la información que guía la búsqueda.